# Lesson 07 — Scale & Cost

You've built the pipeline. Now let's answer the real questions:

- **How do I process 10 image batches without submitting 10 jobs manually?** → Array jobs
- **How much does this actually cost?** → Spot pricing math
- **When is GPU worth it vs just using CPU?** → Rule of thumb

No new ML concepts — this lesson is about production thinking.

## Concept: Batch Array Jobs

An **array job** is one submission that fans out into N parallel copies:

```
submit_job(arrayProperties={"size": 5})
  → 5 containers run simultaneously
  → each gets AWS_BATCH_JOB_ARRAY_INDEX = 0, 1, 2, 3, 4
  → each reads its index and picks which image batch to process
```

Wall-clock time for 5 image batches = wall-clock time for 1 image batch (they run in parallel!).
Cost = 5× a single run.

## Step 1 — Load environment

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path="../../.env")
S3_BUCKET = os.environ["S3_BUCKET"]
print(f"Bucket: {S3_BUCKET}")

## Step 2 — Upload sample image batches

For this demo we'll upload the same `assets/images/` folder under 3 different
prefixes to simulate 3 different image batches. In a real scenario you'd have
different image sets.

In [ ]:
import glob
import os

import boto3

s3 = boto3.client("s3")

local_images = sorted(
    p for ext in ("*.jpg", "*.jpeg", "*.png") for p in glob.glob(os.path.join("assets/images", ext))
)

# Upload the same sample images under 3 different prefixes to simulate 3 image batches
image_prefixes = []
for i in range(1, 4):
    prefix = f"images/batch{i:02d}"
    for path in local_images:
        s3.upload_file(path, S3_BUCKET, f"{prefix}/{os.path.basename(path)}")
    image_prefixes.append(prefix)
    print(f"Uploaded {len(local_images)} images → s3://{S3_BUCKET}/{prefix}/")

print(f"\n{len(image_prefixes)} image batches ready.")

## Step 3 — Submit the array job

In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, "submit_array_job.py", "--image-prefixes"] + image_prefixes,
)
print("Exit code:", result.returncode)

## Step 4 — Cost breakdown

In [ ]:
# Approximate numbers for g4dn.xlarge (NVIDIA T4) Spot in ap-northeast-1
SPOT_PRICE_PER_HOUR  = 0.20    # USD, typical Spot price (range $0.16–0.24)
CAPTION_MINUTES      = 1.0     # per image batch
EMBED_MINUTES        = 0.5     # per image batch
PIPELINE_MINUTES     = CAPTION_MINUTES + EMBED_MINUTES
N_BATCHES            = len(image_prefixes)

# With an array job, all batches run in parallel → wall-clock = 1 pipeline run
# But we pay for N instances
cost_per_instance = (PIPELINE_MINUTES / 60) * SPOT_PRICE_PER_HOUR
total_cost        = cost_per_instance * N_BATCHES
wall_clock_min    = PIPELINE_MINUTES   # parallel, so same as 1

print(f"Image batches processed  : {N_BATCHES}")
print(f"Wall-clock time          : ~{wall_clock_min:.0f} min  (parallel — same as 1 batch!)")
print(f"Cost per instance        : ${cost_per_instance:.4f}")
print(f"Total cost ({N_BATCHES} batches)  : ${total_cost:.4f}")
print()
print(f"At this rate, 100 image batches would cost: ${cost_per_instance * 100:.2f}")

## Step 5 — GPU vs CPU: when does it pay off?

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Hypothetical: caption + embed N images using CPU (350 ms/image) vs GPU (15 ms/image)
n_images    = np.array([10, 50, 100, 500, 1000, 5000, 10000])
cpu_seconds = n_images * 0.35   # 350 ms/image on CPU (BLIP + CLIP combined)
gpu_seconds = n_images * 0.015  # 15 ms/image on GPU (batched)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(n_images, cpu_seconds / 60, label="CPU",  color="steelblue", marker="o")
ax.plot(n_images, gpu_seconds / 60, label="GPU",  color="coral",     marker="o")
ax.set_xscale("log")
ax.set_xlabel("Number of images")
ax.set_ylabel("Time (minutes)")
ax.set_title("GPU vs CPU time for BLIP captioning + CLIP embedding")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Speedup at different scales:")
for n, c, g in zip(n_images, cpu_seconds, gpu_seconds):
    print(f"  {n:6d} images — CPU: {c:6.1f}s  GPU: {g:4.1f}s  Speedup: {c/g:.0f}×")

## 🎉 Course Complete!

Here's what you built — a production-grade GPU pipeline:

```
Images in S3
    ↓  [GPU: generate_captions.py]  → captions in S3
    ↓  [GPU: embed_captions.py]     → CLIP embeddings in S3 Vectors
    ↓  [local: search notebook]     → "find me the photo of a dog in a park" ✅
       [scale: array job]           → process 100 image batches in parallel ✅
```

### What to explore next

| Idea | What to look up |
|------|-----------------|
| Replace S3 Vectors with another managed vector DB | pgvector, Pinecone, OpenSearch |
| Use a larger, more accurate captioning model | BLIP-2, LLaVA |
| Transcribe audio in addition to captioning images | OpenAI Whisper (runs on GPU too!) |
| Orchestrate bigger pipelines | Apache Airflow, AWS Step Functions |
| Train your own embedding model | PyTorch fine-tuning, PEFT/LoRA |